# Plotting and Analyzing the Data from SQL

In [3]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# custom template for the whole notebook
pio.templates["taxi"] = go.layout.Template(
    layout=dict(
        font=dict(family="Inter, Helvetica, Arial, sans-serif",
                  size=13, color="#2b2b2b"),
        paper_bgcolor="white",
        plot_bgcolor="#f7f7f5",
        colorway=["#1b3a4b", "#2c7873", "#c98b3e", "#a63d40", "#6a8d92"],
        title=dict(font=dict(size=19, color="#1b1b1b"), x=0.02),
        margin=dict(l=60, r=30, t=64, b=52),
    )
)
pio.templates.default = "taxi"

# Reusable scale for Vol maps
VOL_SCALE = "Cividis"

# Folder for saving figures 
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Read only so notebook doesnt corrup database 
con = duckdb.connect("nyc_taxi.duckdb", read_only=True)

# Helper function to run a SQL
def run_sql_file(path):
    return con.execute(Path(path).read_text()).df()

# Helper to save figures as dynamic and static
def save_fig(fig, name, scale=2):
    fig.write_image(FIG_DIR / f"{name}.png", scale=scale)  # for the README
    return fig                                             # for inline display

con.execute("SHOW TABLES").df()

,name
0,fact_demand
1,stg_trips


In [ ]:
# Seasonality heatmap: demand by hour and weekday
df = run_sql_file("sql/03_seasonality.sql")

# reshape the 168 long-format rows into a weekday × hour grid
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
                 "Friday", "Saturday", "Sunday"]
grid = (
    df.pivot(index="day_of_week", columns="trip_hour", values="avg_trips_per_day")
      .reindex(weekday_order)          # force calendar order, not alphabetical
)

fig = go.Figure(go.Heatmap(
    z=grid.values,
    x=grid.columns,                    # hours 0–23
    y=grid.index,                      # Monday–Sunday
    colorscale=VOL_SCALE,
    colorbar=dict(title="Avg trips/day"),
    hovertemplate="%{y} · %{x}:00<br>%{z:.0f} trips/day<extra></extra>",
))

fig.update_layout(
    title="NYC Yellow Taxi Demand by Hour and Weekday",
    xaxis=dict(title="Hour of day", tickmode="linear", dtick=2),
    yaxis=dict(title=None, autorange="reversed"),   # Monday at top
    height=430,
)

fig = save_fig(fig, "seasonality_heatmap")
fig.show()

In [ ]:
# Locate 12 busiest zones by total trips
con.execute("""
    SELECT zone_name, SUM(trips) AS total
    FROM fact_demand GROUP BY zone_name
    ORDER BY total DESC LIMIT 12
""").df()

,zone_name,total
0,Upper East Side South,1011207.0
1,Upper East Side North,918262.0
2,Midtown Center,893419.0
3,JFK Airport,843380.0
4,Midtown East,670034.0
5,Penn Station/Madison Sq West,668389.0
6,Lincoln Square East,656686.0
7,Times Sq/Theatre District,629524.0
8,East Village,582923.0
9,Upper West Side South,578870.0


In [14]:
# Raw daily demand vs. 7-day moving average, for a few zones
df = run_sql_file("sql/04_moving_avg.sql")

# pick a handful of busy, recognizable zones to keep the chart readable
zones = ["JFK Airport", "Midtown Center", "Upper East Side South",
         "Upper East Side North"]
palette = ["#1b3a4b", "#2c7873", "#c98b3e", "#a63d40"]

fig = go.Figure()

for zone, color in zip(zones, palette):
    d = df[df["zone_name"] == zone].sort_values("trip_date")
    # faint raw series
    fig.add_trace(go.Scatter(
        x=d["trip_date"], y=d["daily_trips"],
        mode="lines", line=dict(color=color, width=1),
        opacity=0.25, showlegend=False, hoverinfo="skip",
    ))
    # bold smoothed series
    fig.add_trace(go.Scatter(
        x=d["trip_date"], y=d["moving_avg_7d"],
        mode="lines", line=dict(color=color, width=2.5),
        name=zone,
        hovertemplate=f"{zone}<br>%{{x|%b %d}}<br>%{{y:.0f}} trips/day (7d avg)<extra></extra>",
    ))

fig.update_layout(
    title="Daily Demand vs. 7-Day Moving Average",
    xaxis=dict(title=None),
    yaxis=dict(title="Trips per day"),
    height=460,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)

fig = save_fig(fig, "moving_average_trends")
fig.show()

In [18]:
# Most volatile zone-hours by coefficient of variation
# 05_volatility.sql holds two queries; run just the first (total CV) inline.
vol_sql = Path("sql/05_volatility.sql").read_text().split(";")[0]
df = con.execute(vol_sql).df()

top = df.head(15).copy()
# a readable label per zone-hour, e.g. "JFK Airport · 04:00"
top["zone_short"] = top["zone_name"].str.replace(
    "Flushing Meadows-Corona Park", "Flushing Meadows", regex=False)
top["label"] = top["zone_short"] + " · " + top["trip_hour"].map(lambda h: f"{h:02d}:00")
top = top.sort_values("coeff_of_variation")   # ascending → largest ends up on top

fig = go.Figure(go.Bar(
    x=top["coeff_of_variation"],
    y=top["label"],
    orientation="h",
    marker=dict(
        color=top["coeff_of_variation"],
        colorscale=VOL_SCALE,
        showscale=False,
    ),
    hovertemplate="%{y}<br>CV %{x:.2f}"
                  "<br>mean %{customdata[0]:.1f} trips/hr<extra></extra>",
    customdata=top[["mean_trips"]].values,
))

fig.update_layout(
    title="Most Volatile Zone-Hours (Coefficient of Variation)",
    xaxis=dict(title="Coefficient of variation (σ / μ)"),
    yaxis=dict(title=None),
    height=560,
    margin=dict(l=220),   # room for the long zone-hour labels
)

fig = save_fig(fig, "volatility_ranking")
fig.show()

In [19]:
# Volatility decomposition: total CV vs. within-weekday (seasonally adjusted) CV
# Run BOTH queries from 05, then join them on zone-hour to compare the two CVs.
stmts = [s for s in Path("sql/05_volatility.sql").read_text().split(";") if s.strip()]
total_cv   = con.execute(stmts[0]).df()   # query 1: total CV
withinwd_cv = con.execute(stmts[1]).df()  # query 2: within-weekday CV

# join the two measures on the zone-hour key
merged = total_cv.merge(
    withinwd_cv[["zone_name", "trip_hour", "within_wd_cv"]],
    on=["zone_name", "trip_hour"], how="inner",
)

# focus on zone-hours with real activity, and take the most volatile by total CV
merged = merged[merged["mean_trips"] >= 3]
top = merged.sort_values("coeff_of_variation", ascending=False).head(15).copy()
top["label"] = top["zone_name"] + " · " + top["trip_hour"].map(lambda h: f"{h:02d}:00")
top = top.sort_values("coeff_of_variation")   # ascending → most volatile on top

fig = go.Figure()

# total CV — the fainter, "raw" measure
fig.add_trace(go.Bar(
    x=top["coeff_of_variation"], y=top["label"], orientation="h",
    name="Total CV", marker=dict(color="#c98b3e"), opacity=0.55,
    hovertemplate="%{y}<br>Total CV %{x:.2f}<extra></extra>",
))
# within-weekday CV — the bold, seasonally-adjusted measure
fig.add_trace(go.Bar(
    x=top["within_wd_cv"], y=top["label"], orientation="h",
    name="Within-weekday CV", marker=dict(color="#1b3a4b"),
    hovertemplate="%{y}<br>Within-weekday CV %{x:.2f}<extra></extra>",
))

fig.update_layout(
    title="Volatility Decomposition: Total vs. Seasonally-Adjusted",
    xaxis=dict(title="Coefficient of variation (σ / μ)"),
    yaxis=dict(title=None),
    barmode="group",
    height=600,
    margin=dict(l=260, t=90),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)

fig = save_fig(fig, "volatility_decomposition")
fig.show()

In [ ]:
#3D volatility surface for a single zone: CV by hour × weekday
ZONE = "Upper East Side South"   # swap for any zone 

# per (weekday, hour) mean, sd, and CV 
surface_sql = f"""
WITH calendar AS (SELECT DISTINCT trip_date FROM fact_demand),
zone_hours AS (
    SELECT DISTINCT pickup_zone_id, zone_name, trip_hour
    FROM fact_demand WHERE zone_name = '{ZONE}'
),
full_grid AS (
    SELECT zh.trip_hour, c.trip_date,
           DAYNAME(c.trip_date) AS day_of_week,
           COALESCE(f.trips, 0) AS trips
    FROM zone_hours zh
    CROSS JOIN calendar c
    LEFT JOIN fact_demand f
        ON f.pickup_zone_id = zh.pickup_zone_id
       AND f.trip_hour = zh.trip_hour
       AND f.trip_date = c.trip_date
)
SELECT day_of_week, trip_hour,
       STDDEV_SAMP(trips) / NULLIF(AVG(trips), 0) AS cv
FROM full_grid
GROUP BY day_of_week, trip_hour
"""
df = con.execute(surface_sql).df()

# reshape to a weekday × hour grid (same pivot + reindex pattern as the heatmap)
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
                 "Friday", "Saturday", "Sunday"]
grid = (df.pivot(index="day_of_week", columns="trip_hour", values="cv")
          .reindex(weekday_order))

fig = go.Figure(go.Surface(
    z=grid.values,
    x=grid.columns,            # hours 0–23
    y=grid.index,              # Monday–Sunday
    colorscale=VOL_SCALE,
    colorbar=dict(title="CV"),
    hovertemplate="%{y} · %{x}:00<br>CV %{z:.2f}<extra></extra>",
))

fig.update_layout(
    title=f"Demand Volatility Surface — {ZONE}",
    height=620,
    margin=dict(l=0, r=0, t=60, b=0),
    scene=dict(
        xaxis_title="Hour of day",
        yaxis_title=None,
        zaxis_title="Coefficient of variation",
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)),  # starting viewing angle
    ),
)

fig = save_fig(fig, "volatility_surface")
fig.show()

In [23]:
diag_sql = """
WITH calendar AS (SELECT DISTINCT trip_date FROM fact_demand),
zone_hours AS (
    SELECT DISTINCT pickup_zone_id, trip_hour
    FROM fact_demand WHERE zone_name = 'JFK Airport'
),
full_grid AS (
    SELECT zh.trip_hour, c.trip_date,
           DAYNAME(c.trip_date) AS day_of_week,
           COALESCE(f.trips, 0) AS trips
    FROM zone_hours zh
    CROSS JOIN calendar c
    LEFT JOIN fact_demand f
        ON f.pickup_zone_id = zh.pickup_zone_id
       AND f.trip_hour = zh.trip_hour
       AND f.trip_date = c.trip_date
)
SELECT day_of_week, trip_hour,
       COUNT(*)                                   AS num_days,
       ROUND(AVG(trips), 2)                       AS mean_trips,
       ROUND(STDDEV_SAMP(trips), 2)               AS sd_trips,
       ROUND(STDDEV_SAMP(trips)/NULLIF(AVG(trips),0), 2) AS cv,
       MAX(trips)                                 AS max_trips,
       ROUND(MEDIAN(trips), 1)                    AS median_trips
FROM full_grid
WHERE trip_hour IN (0, 1, 2, 3)
GROUP BY day_of_week, trip_hour
ORDER BY trip_hour, cv DESC
"""
con.execute(diag_sql).df()

,day_of_week,trip_hour,num_days,mean_trips,sd_trips,cv,max_trips,median_trips
0,Thursday,0,26,163.96,85.15,0.52,410,145.5
1,Sunday,0,26,171.96,74.28,0.43,430,163.0
2,Saturday,0,26,161.62,68.52,0.42,394,157.0
3,Wednesday,0,26,173.81,70.80,0.41,298,166.0
4,Monday,0,26,260.92,101.02,0.39,415,266.5
5,Tuesday,0,26,248.15,95.59,0.39,436,235.5
6,Friday,0,26,211.31,56.33,0.27,350,201.5
7,Thursday,1,26,98.38,83.31,0.85,433,78.5
8,Saturday,1,26,74.15,52.77,0.71,272,62.5
9,Monday,1,26,150.00,96.36,0.64,360,129.5


In [24]:
con.execute("""
    SELECT trip_date, DAYNAME(trip_date) AS dow, SUM(trips) AS late_night_trips
    FROM fact_demand
    WHERE zone_name = 'JFK Airport' AND trip_hour IN (0,1,2,3)
      AND DAYNAME(trip_date) IN ('Thursday','Friday','Saturday')
    GROUP BY trip_date
    ORDER BY trip_date
""").df()

,trip_date,dow,late_night_trips
0,2025-12-04,Thursday,264.0
1,2025-12-05,Friday,273.0
2,2025-12-06,Saturday,234.0
3,2025-12-11,Thursday,277.0
4,2025-12-12,Friday,464.0
...,...,...,...
73,2026-05-22,Friday,339.0
74,2026-05-23,Saturday,218.0
75,2026-05-28,Thursday,413.0
76,2026-05-29,Friday,308.0


In [ ]:
#Week-over-week demand growth for selected zones
df = run_sql_file("sql/06_growth.sql")

zones = ["Upper East Side South", "Midtown Center", "JFK Airport",
         "Penn Station/Madison Sq West"]
palette = ["#1b3a4b", "#2c7873", "#c98b3e", "#a63d40"]

# drop the partial first/last weeks per zone
def trim_partial_weeks(g):
    return g.sort_values("week_start").iloc[1:-1]

fig = go.Figure()
fig.add_hline(y=0, line=dict(color="#999", width=1, dash="dot"))  # zero-growth reference

for zone, color in zip(zones, palette):
    d = trim_partial_weeks(df[df["zone_name"] == zone])
    fig.add_trace(go.Scatter(
        x=d["week_start"], y=d["wow_growth_pct"],
        mode="lines+markers", name=zone,
        line=dict(color=color, width=2),
        marker=dict(size=5),
        hovertemplate=f"{zone}<br>week of %{{x|%b %d}}"
                      f"<br>%{{y:+.1f}}%<extra></extra>",
    ))

fig.update_layout(
    title="Week-over-Week Demand Growth",
    xaxis=dict(title=None),
    yaxis=dict(title="WoW growth (%)", ticksuffix="%", zeroline=False),
    height=470,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)

fig = save_fig(fig, "wow_growth")
fig.show()